# GoEmotions Multi-Label Training Pipeline (RoBERTa)

This notebook trains and evaluates a RoBERTa model for multi-label emotion classification.

What this pipeline handles:
- Multi-label targets (each sample can have multiple emotions)
- Stratified train/validation/test split adapted for multi-label data
- Fine-tuning with `transformers` + `torch`
- Evaluation with micro/macro precision, recall, F1, subset accuracy, and Hamming loss

Dataset expected path: `../dataset/go_emotions_dataset.csv`

## 1) Imports And Setup

We only use libraries already listed in `requirements.txt`.

Device priority:
- Apple Silicon GPU via MPS (Metal)
- CUDA GPU
- CPU fallback

In [35]:
import os

# Must be set BEFORE `import torch` so the MPS backend picks it up.
# Allows unsupported ops to fall back to CPU instead of crashing, while
# keeping the rest of the computation on the Apple GPU.
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import random
import warnings
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    precision_recall_fscore_support,
    accuracy_score,
    hamming_loss,
)
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Prefer Apple GPU (MPS) on Apple Silicon, then CUDA, otherwise CPU.
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

print("Using device:", device)
print("torch:", torch.__version__)
print("MPS available:", torch.backends.mps.is_available(),
      "| MPS built:", torch.backends.mps.is_built())
if device.type == "mps":
    print("MPS backend enabled for Apple Silicon GPU acceleration.")
    print("PYTORCH_ENABLE_MPS_FALLBACK =", os.environ.get("PYTORCH_ENABLE_MPS_FALLBACK"))


Using device: mps
torch: 2.13.0
MPS available: True | MPS built: True
MPS backend enabled for Apple Silicon GPU acceleration.
PYTORCH_ENABLE_MPS_FALLBACK = 1


## 2) Load Data And Build Multi-Label Targets

The dataset has one binary column per emotion. We convert those columns into a multi-hot label vector for each sample.

In [36]:
DATA_PATH = "../datasets/go_emotions_dataset.csv"
TEXT_COL = "text"
META_COLS = ["id", "example_very_unclear"]

raw_df = pd.read_csv(DATA_PATH)

label_cols = [
    c for c in raw_df.columns
    if c not in META_COLS + [TEXT_COL]
]

# Keep only rows with at least one active label for supervised multi-label training.
raw_df = raw_df[raw_df[label_cols].sum(axis=1) > 0].reset_index(drop=True)

print("Filtered dataset shape:", raw_df.shape)
print("Number of label columns:", len(label_cols))

texts = raw_df[TEXT_COL].astype(str).tolist()
Y = raw_df[label_cols].astype(np.float32).values

print("Label matrix shape:", Y.shape)
print("Average active labels per sample:", Y.sum(axis=1).mean())

Filtered dataset shape: (207814, 31)
Number of label columns: 28
Label matrix shape: (207814, 28)
Average active labels per sample: 1.2007324


## 3) Multi-Label Stratified Train/Validation/Test Split

True iterative multi-label stratification usually needs external packages not listed in this project.

Here we use a practical approximation:
1. Convert each sample's active label set to a label-signature string.
2. Collapse very rare signatures into an `__OTHER__` bucket.
3. Apply stratified splitting twice (train vs temp, then temp into validation/test).

This preserves label-combination diversity better than random splitting while staying within available dependencies.

In [37]:
def make_signature(y_row, label_names):
    active = [label_names[i] for i, v in enumerate(y_row) if v > 0.5]
    return "|".join(active) if active else "__NONE__"


def build_stratify_keys(Y_array, label_names, min_count=20):
    signatures = [make_signature(row, label_names) for row in Y_array]
    counts = pd.Series(signatures).value_counts()
    keys = [sig if counts[sig] >= min_count else "__OTHER__" for sig in signatures]
    return np.array(keys)


TEST_SIZE = 0.10
VAL_SIZE = 0.10
MIN_SIGNATURE_COUNT = 20

all_idx = np.arange(len(texts))
strat_keys = build_stratify_keys(Y, label_cols, min_count=MIN_SIGNATURE_COUNT)

train_idx, temp_idx = train_test_split(
    all_idx,
    test_size=TEST_SIZE + VAL_SIZE,
    random_state=SEED,
    shuffle=True,
    stratify=strat_keys,
)

# Split the temp set into validation and test using the same strategy.
Y_temp = Y[temp_idx]
strat_keys_temp = build_stratify_keys(Y_temp, label_cols, min_count=max(5, MIN_SIGNATURE_COUNT // 2))

val_ratio_of_temp = VAL_SIZE / (VAL_SIZE + TEST_SIZE)
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=1 - val_ratio_of_temp,
    random_state=SEED,
    shuffle=True,
    stratify=strat_keys_temp,
)

print(f"Train size: {len(train_idx)}")
print(f"Validation size: {len(val_idx)}")
print(f"Test size: {len(test_idx)}")

Train size: 166251
Validation size: 20781
Test size: 20782


## 4) Build Datasets And Tokenize

We tokenize text with RoBERTa tokenizer and wrap the encoded tensors in a custom PyTorch Dataset.

Local model note:
- You can point to a model directory OR directly to a `.safetensors` weights file.
- If a file path is used, the notebook uses its parent directory for config/tokenizer files.
- Required local files typically include: `config.json`, tokenizer files, and model weights.

In [38]:
MODEL_NAME = "roberta-base"
MODEL_SOURCE = os.getenv("ROBERTA_MODEL_PATH", "../models/roberta-base.safetensors")
LOCAL_FILES_ONLY = bool(int(os.getenv("HF_LOCAL_FILES_ONLY", "1")))
MAX_LENGTH = 128

if MODEL_SOURCE.endswith(".safetensors"):
    MODEL_WEIGHTS_FILE = MODEL_SOURCE
    MODEL_PATH = os.path.dirname(os.path.abspath(MODEL_SOURCE))
else:
    MODEL_PATH = MODEL_SOURCE
    MODEL_WEIGHTS_FILE = None

TOKENIZER_PATH = os.getenv("ROBERTA_TOKENIZER_PATH", MODEL_PATH)

train_df = raw_df.iloc[train_idx].reset_index(drop=True)
val_df = raw_df.iloc[val_idx].reset_index(drop=True)
test_df = raw_df.iloc[test_idx].reset_index(drop=True)

# Try tokenizer from explicit local path first, then local cache under model name.
try:
    tokenizer = AutoTokenizer.from_pretrained(
        TOKENIZER_PATH,
        local_files_only=LOCAL_FILES_ONLY,
        use_fast=True,
    )
except Exception:
    try:
        tokenizer = AutoTokenizer.from_pretrained(
            MODEL_NAME,
            local_files_only=True,
            use_fast=True,
        )
        print("Tokenizer loaded from local HF cache for roberta-base.")
    except Exception as e:
        raise RuntimeError(
            "Failed to load tokenizer from local files. Place tokenizer assets in a local directory "
            "(tokenizer.json + vocab files + tokenizer_config.json) and set ROBERTA_TOKENIZER_PATH."
        ) from e


def encode_texts(text_list):
    return tokenizer(
        text_list,
        truncation=True,
        max_length=MAX_LENGTH,
    )


train_encodings = encode_texts(train_df[TEXT_COL].astype(str).tolist())
val_encodings = encode_texts(val_df[TEXT_COL].astype(str).tolist())
test_encodings = encode_texts(test_df[TEXT_COL].astype(str).tolist())

train_labels = train_df[label_cols].astype(np.float32).values
dev_labels = val_df[label_cols].astype(np.float32).values
test_labels = test_df[label_cols].astype(np.float32).values


class MultiLabelTextDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)
        return item


train_ds = MultiLabelTextDataset(train_encodings, train_labels)
val_ds = MultiLabelTextDataset(val_encodings, dev_labels)
test_ds = MultiLabelTextDataset(test_encodings, test_labels)

print(f"Model source: {MODEL_SOURCE}")
print(f"Model path: {MODEL_PATH}")
print(f"Tokenizer path: {TOKENIZER_PATH}")
print(f"Weights file: {MODEL_WEIGHTS_FILE}")
print(f"Local files only: {LOCAL_FILES_ONLY}")
print(f"Train dataset size: {len(train_ds)}")
print(f"Validation dataset size: {len(val_ds)}")
print(f"Test dataset size: {len(test_ds)}")

Model source: ../models/roberta-base.safetensors
Model path: /Users/mac-SAITSI15/Desktop/goemotions/models
Tokenizer path: /Users/mac-SAITSI15/Desktop/goemotions/models
Weights file: ../models/roberta-base.safetensors
Local files only: True
Train dataset size: 166251
Validation dataset size: 20781
Test dataset size: 20782


## 5) Define Model, Dataloaders, And Multi-Label Metrics

For multi-label classification, RoBERTa uses sigmoid outputs per label and Binary Cross-Entropy loss.

Thresholding strategy:
- Convert probabilities to binary predictions with a fixed threshold (default 0.5).

In [39]:
THRESHOLD = 0.5
LR = 2e-5
EPOCHS = 2
TRAIN_BATCH_SIZE = 32
EVAL_BATCH_SIZE = 64
GRAD_ACCUM_STEPS = 2  # Effective batch size = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS = 64
FREEZE_LAYERS = 8  # Freeze embeddings + first 8 of 12 encoder layers
OUTPUT_DIR = "../artifacts/roberta-goemotions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

id2label = {i: lbl for i, lbl in enumerate(label_cols)}
label2id = {lbl: i for i, lbl in enumerate(label_cols)}


def load_local_or_pretrained_model():
    # If an explicit safetensors file is provided, try manual state_dict loading.
    if MODEL_WEIGHTS_FILE and os.path.isfile(MODEL_WEIGHTS_FILE):
        try:
            from safetensors.torch import load_file

            config = AutoConfig.from_pretrained(
                MODEL_PATH,
                local_files_only=LOCAL_FILES_ONLY,
            )
            config.num_labels = len(label_cols)
            config.problem_type = "multi_label_classification"
            config.id2label = id2label
            config.label2id = label2id

            # Disable attention dropout so MPS scaled_dot_product_attention
            # works under autocast (MPS SDPA does not support dropout).
            # Hidden-layer and classifier dropout still provide regularization.
            if device.type == "mps":
                config.attention_probs_dropout_prob = 0.0

            # Build on CPU first, load weights, THEN move to device.
            # This avoids any risk of parameters being re-allocated on CPU
            # by load_state_dict after an early .to(device) call.
            local_model = AutoModelForSequenceClassification.from_config(config)
            state_dict = load_file(MODEL_WEIGHTS_FILE)
            local_model.load_state_dict(state_dict, strict=False)
            local_model.to(device)
            return local_model
        except Exception as e:
            raise RuntimeError(
                "Failed to load model from safetensors file. Ensure local config/tokenizer files "
                "exist in the same directory as ROBERTA_MODEL_PATH."
            ) from e

    try:
        hf_model = AutoModelForSequenceClassification.from_pretrained(
            MODEL_PATH,
            local_files_only=LOCAL_FILES_ONLY,
            num_labels=len(label_cols),
            problem_type="multi_label_classification",
            id2label=id2label,
            label2id=label2id,
            attention_probs_dropout_prob=0.0 if device.type == "mps" else 0.1,
        )
        hf_model.to(device)
        return hf_model
    except Exception as e:
        raise RuntimeError(
            "Failed to load RoBERTa model weights from local files. Ensure ROBERTA_MODEL_PATH points "
            "to a local model directory (or safetensors file with matching local config files)."
        ) from e


model = load_local_or_pretrained_model()

# --- Freeze lower layers for faster training ---
# Freeze embeddings
for param in model.roberta.embeddings.parameters():
    param.requires_grad = False
# Freeze first FREEZE_LAYERS encoder layers
for layer in model.roberta.encoder.layer[:FREEZE_LAYERS]:
    for param in layer.parameters():
        param.requires_grad = False

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"Trainable params: {trainable_params:,} / {total_params:,} "
      f"({100 * trainable_params / total_params:.1f}%)")

# Sanity-check: confirm the model actually lives on the intended device.
_model_device = next(model.parameters()).device
print(f"Requested device: {device} | Model parameters device: {_model_device}")
assert _model_device.type == device.type, (
    f"Model is on {_model_device} but expected {device}. "
    "Training would silently run on the wrong device."
)


def collate_fn(batch):
    features = [{"input_ids": x["input_ids"], "attention_mask": x["attention_mask"]} for x in batch]
    padded = tokenizer.pad(features, padding=True, return_tensors="pt")
    labels = torch.stack([x["labels"] for x in batch]).float()
    padded["labels"] = labels
    return padded


loader_workers = 0 if device.type == "mps" else 2
pin_memory = device.type == "cuda"

train_loader = DataLoader(
    train_ds,
    batch_size=TRAIN_BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=loader_workers,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=loader_workers,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_ds,
    batch_size=EVAL_BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=loader_workers,
    pin_memory=pin_memory,
)

# Only pass trainable parameters to the optimizer.
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()), lr=LR
)
num_training_steps = (EPOCHS * len(train_loader)) // GRAD_ACCUM_STEPS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)


def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def compute_metrics_from_logits(logits, labels, threshold=THRESHOLD):
    probs = sigmoid(logits)
    preds = (probs >= threshold).astype(int)
    labels = labels.astype(int)

    p_micro, r_micro, f1_micro, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="micro",
        zero_division=0,
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="macro",
        zero_division=0,
    )

    subset_acc = accuracy_score(labels, preds)
    ham = hamming_loss(labels, preds)

    return {
        "micro_precision": p_micro,
        "micro_recall": r_micro,
        "micro_f1": f1_micro,
        "macro_precision": p_macro,
        "macro_recall": r_macro,
        "macro_f1": f1_macro,
        "subset_accuracy": subset_acc,
        "hamming_loss": ham,
    }


def evaluate_loader(model_obj, data_loader):
    model_obj.eval()
    total_loss = 0.0
    all_logits = []
    all_labels = []

    with torch.no_grad(), torch.amp.autocast(device.type):
        for batch in tqdm(data_loader, desc="Evaluating", leave=False):
            batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
            outputs = model_obj(**batch)
            loss = outputs.loss
            logits = outputs.logits

            total_loss += loss.item()
            all_logits.append(logits.detach().float().cpu().numpy())
            all_labels.append(batch["labels"].detach().cpu().numpy())

    avg_loss = total_loss / max(1, len(data_loader))
    logits_np = np.concatenate(all_logits, axis=0)
    labels_np = np.concatenate(all_labels, axis=0)
    metrics = compute_metrics_from_logits(logits_np, labels_np)
    metrics["loss"] = avg_loss
    return metrics, logits_np, labels_np


print("Model and dataloaders ready.")
print(f"DataLoader workers: {loader_workers}, pin_memory: {pin_memory}")
print(f"Gradient accumulation steps: {GRAD_ACCUM_STEPS} (effective batch size: {TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS})")
print(f"Frozen layers: embeddings + first {FREEZE_LAYERS} encoder layers")
if device.type == "mps":
    print("Attention dropout disabled for MPS SDPA compatibility.")


Trainable params: 28,963,612 / 124,667,164 (23.2%)
Requested device: mps | Model parameters device: mps:0
Model and dataloaders ready.
DataLoader workers: 0, pin_memory: False
Gradient accumulation steps: 2 (effective batch size: 64)
Frozen layers: embeddings + first 8 encoder layers
Attention dropout disabled for MPS SDPA compatibility.


## 6) Train And Validate

This runs epoch-based fine-tuning and tracks validation performance after each epoch.

In [40]:
best_val_f1 = -1.0
best_model_path = os.path.join(OUTPUT_DIR, "best_model.pt")
history = []

# Log once which device the model actually lives on.
print("Training device (model):", next(model.parameters()).device)
print(f"Mixed precision (autocast) enabled for '{device.type}'")

for epoch in range(1, EPOCHS + 1):
    model.train()
    # Accumulate loss as a device tensor to avoid a host<->device sync every step.
    epoch_loss = torch.zeros((), device=device)
    epoch_steps = 0
    last_reported_loss = None

    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS} - Train", leave=False)
    for step, batch in enumerate(progress):
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}

        # Mixed precision forward pass
        with torch.amp.autocast(device.type):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM_STEPS  # Scale loss for accumulation

        loss.backward()

        epoch_loss += loss.detach() * GRAD_ACCUM_STEPS
        epoch_steps += 1

        # Optimizer step every GRAD_ACCUM_STEPS
        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        # Only sync loss to CPU occasionally for the progress bar.
        if step % 20 == 0:
            last_reported_loss = (loss.detach() * GRAD_ACCUM_STEPS).item()
            progress.set_postfix(train_loss=last_reported_loss)

    avg_train_loss = (epoch_loss / max(1, epoch_steps)).item()
    val_metrics, _, _ = evaluate_loader(model, val_loader)

    row = {
        "epoch": epoch,
        "train_loss": avg_train_loss,
        "val_loss": val_metrics["loss"],
        "val_micro_f1": val_metrics["micro_f1"],
        "val_macro_f1": val_metrics["macro_f1"],
    }
    history.append(row)

    print(f"Epoch {epoch}")
    print({k: round(v, 4) if isinstance(v, float) else v for k, v in row.items()})

    if val_metrics["micro_f1"] > best_val_f1:
        best_val_f1 = val_metrics["micro_f1"]
        torch.save(model.state_dict(), best_model_path)

history_df = pd.DataFrame(history)
print("\nTraining history:")
display(history_df)
print("Best validation micro F1:", round(best_val_f1, 4))
print("Saved best model to:", best_model_path)


Training device (model): mps:0
Mixed precision (autocast) enabled for 'mps'


Epoch 1
{'epoch': 1, 'train_loss': 0.1655, 'val_loss': 0.1187, 'val_micro_f1': 0.2841, 'val_macro_f1': 0.1588}


Epoch 2
{'epoch': 2, 'train_loss': 0.1169, 'val_loss': 0.1157, 'val_micro_f1': 0.3116, 'val_macro_f1': 0.2039}

Training history:


,epoch,train_loss,val_loss,val_micro_f1,val_macro_f1
0,1,0.165491,0.118746,0.284142,0.158839
1,2,0.116921,0.115680,0.311594,0.203926


Best validation micro F1: 0.3116
Saved best model to: ../artifacts/roberta-goemotions/best_model.pt


## 7) Final Test Evaluation

After selecting the best checkpoint on validation, evaluate once on the held-out test split.

In [41]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.to(device)

test_metrics, test_logits, test_labels = evaluate_loader(model, test_loader)

print("Test metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

Test metrics:
micro_precision: 0.6557
micro_recall: 0.2072
micro_f1: 0.3149
macro_precision: 0.4426
macro_recall: 0.1597
macro_f1: 0.2061
subset_accuracy: 0.1896
hamming_loss: 0.0387
loss: 0.1148


## 8) Optional: Per-Label Report On Test Set

This table shows precision, recall, and F1 for each emotion label.

In [42]:
test_probs = 1.0 / (1.0 + np.exp(-test_logits))
test_preds = (test_probs >= THRESHOLD).astype(int)

p, r, f1, support = precision_recall_fscore_support(
    test_labels.astype(int),
    test_preds,
    average=None,
    zero_division=0,
)

per_label_df = pd.DataFrame(
    {
        "label": label_cols,
        "precision": p,
        "recall": r,
        "f1": f1,
        "support": support,
    }
).sort_values("f1", ascending=False)

per_label_df.head(28)

,label,precision,recall,f1,support
15,gratitude,0.878486,0.757732,0.813653,1164
18,love,0.679898,0.651534,0.665414,815
1,amusement,0.590755,0.568947,0.579646,921
0,admiration,0.638498,0.472769,0.543276,1726
14,fear,0.696429,0.252427,0.370546,309
27,neutral,0.643071,0.233273,0.342357,5530
20,optimism,0.613115,0.213470,0.316681,876
24,remorse,0.548077,0.222656,0.316667,256
17,joy,0.579767,0.184864,0.280339,806
25,sadness,0.591160,0.158284,0.249708,676
